[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Optimizer077/jev-learning-course/blob/main/notebooks/04_workflows_and_related_models.ipynb)

# 4 · Put decisions inside a software workflow

**Path:** Beginner → builder  
**Before you start:** Lesson 01; understand an if statement

[Course home](../README.md) · [Course outline](../docs/COURSE.md) · [Setup help](../docs/SETUP.md) · [Glossary](../docs/GLOSSARY.md)

## The idea before the code

Let the model make a small judgment. Let ordinary code combine judgments, check permissions, and choose the next step.

**Your first pass:** Work through the four fictional tickets. Then read the question rewrites and dependency example.

### Words you need for this lesson

| Word | Everyday meaning |
|---|---|
| **Workflow** | The sequence of judgments and actions in a program. |
| **Permission** | Whether the program is allowed to perform an action. |
| **Dependency** | Information one step must receive from another step. |

## Goal
Combine narrow judgments with ordinary code, and understand how Jev relates to other model families.

TypeSafe documents evaluation of multiple questions against one shared state in parallel and isolation.
This is an execution/interface property, **not a guarantee that the real-world events are statistically
independent**. [Introduction](https://docs.typesafe.ai/introduction).

## Setup
All answers below are invented. The code only returns route names; it does not send messages or execute actions.

In [1]:
# Find the included teaching helpers from the course folder.
from pathlib import Path
import sys
REQUIRED_FILES = ("src/lab_core.py", "src/tutorial_utils.py", "data/tickets.json")
COURSE_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                    if all((p / name).is_file() for name in REQUIRED_FILES)), None)
# Colab opens a single notebook; fetch its companion code and fictional data.
if COURSE_ROOT is None:
    try:
        import google.colab
    except ImportError:
        pass
    else:
        import subprocess
        COURSE_ROOT = Path("/content/jev-learning-course")
        if COURSE_ROOT.exists() and not all((COURSE_ROOT / name).is_file() for name in REQUIRED_FILES):
            raise FileNotFoundError("The cached Colab course is incomplete. Start a fresh runtime and run all again.")
        if not COURSE_ROOT.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/Optimizer077/jev-learning-course.git",
                            str(COURSE_ROOT)], check=True)
if COURSE_ROOT is None:
    raise FileNotFoundError("Extract the complete course and open it from its folder.")
if str(COURSE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
from tutorial_utils import setup, show_figure, table, BLUE, ORANGE, GREY
setup()

## Steps
### 1. Decompose a broad judgment
Instead of “handle this ticket,” ask separately about topic, blocking impact, and missing information.
Then implement policy in code. A later question that needs an earlier answer requires another stage;
putting it in the same parallel call does not create that dependency.

In [2]:
tickets = [
    {'id': 'A', 'queue': 'technical', 'topic_max_p': 0.94, 'blocked_p': 0.90, 'missing_p': 0.05},
    {'id': 'B', 'queue': 'billing', 'topic_max_p': 0.91, 'blocked_p': 0.10, 'missing_p': 0.10},
    {'id': 'C', 'queue': 'technical', 'topic_max_p': 0.52, 'blocked_p': 0.70, 'missing_p': 0.15},
    {'id': 'D', 'queue': 'other', 'topic_max_p': 0.88, 'blocked_p': 0.20, 'missing_p': 0.85},
]

def route(ticket):
    # These arbitrary teaching thresholds use probabilities, not API confidence.
    if ticket['missing_p'] >= 0.60:
        return 'collect_more_information'
    if ticket['topic_max_p'] < 0.85 or ticket['queue'] == 'other':
        return 'human_review'
    if ticket['queue'] == 'technical' and ticket['blocked_p'] >= 0.80:
        return 'priority_technical_queue'
    return ticket['queue'] + '_queue'

table(['Ticket', 'Suggested route'], [(ticket['id'], route(ticket)) for ticket in tickets])

Ticket,Suggested route
A,priority_technical_queue
B,billing_queue
C,human_review
D,collect_more_information


### 2. Do not multiply unrelated marginal probabilities blindly
If $P(A)=0.8$ and $P(B)=0.7$, then $P(A\cap B)$ is between
$\max(0,P(A)+P(B)-1)$ and $\min(P(A),P(B))$. Multiplication gives 0.56 only under independence.
Parallel model evaluation does not establish that assumption.

In [3]:
p_a, p_b = 0.8, 0.7
lower, upper = max(0, p_a + p_b - 1), min(p_a, p_b)
print(f'Possible joint probability: {lower:.2f} to {upper:.2f}')
print(f'If independent: {p_a * p_b:.2f}')

Possible joint probability: 0.50 to 0.70
If independent: 0.56


### 3. Compare related approaches
These are conceptual distinctions, not benchmark rankings.

| Approach | Typical input → output | Relationship to this lesson |
|---|---|---|
| Deterministic rules | Explicit fields → computed result | Best for exact arithmetic, permissions, and known conditions |
| Supervised classifier | Features/text → fixed training labels | Notebook 2; new classes normally require adapting training or the head |
| Encoder with a classification head | Text representation → labels | Can also score categories directly without generating prose |
| Embeddings and similarity | Text → vectors → similarity | Useful for retrieval; cosine similarity is not automatically a probability |
| Pairwise/cross-encoder scorer | Query and candidate → relevance score | Useful for reranking; score calibration requires separate evaluation |
| Generative language model | Context → token sequence | Can generate text and code; constrained decoding can enforce output schemas |
| Jev's documented interface | State + typed questions → decisions and probabilities | Natural-language criteria define the bounded judgments for each request |

The existence of typed output alone does not prove a novel architecture: classifiers and constrained
decoders already constrain outputs. Jev's claimed contribution combines a decision-oriented interface,
training, and execution. The exact comparison needs task-matched measurements, not labels.

An LLM can use constrained decoding and return valid JSON. That addresses syntax, not automatically
truth or calibration. Conversely, a classifier's softmax numbers are not automatically calibrated.

### 4. What should stay in code?
TypeSafe lists weaknesses in exact arithmetic, long irrelevant context, adversarial text, and consistency
between differently phrased questions. Use explicit criteria, filter evidence, and enforce required
identities in software. It also warns against treating rubric scores as precise numerical reconstruction.
[Known limitations](https://docs.typesafe.ai/model-jaggedness/jev-1.13).

In [4]:
# Exact money arithmetic belongs in code; integers represent cents.
unit_price_cents, quantity = 1299, 3
total_cents = unit_price_cents * quantity
print(f'Exact total: {total_cents / 100:.2f}')

# If these are exact complements in our application, derive the complement.
refund_requested_p = 0.72
no_refund_requested_p = 1 - refund_requested_p
print('Complement computed in code:', round(no_refund_requested_p, 2))

Exact total: 38.97
Complement computed in code: 0.28


> **Optional deeper practice.** You can stop reading here on your first pass. If you run the notebook, use Run All so the later checks still have their inputs.

### 5. Rewrite vague questions into testable contracts
The criteria should tell a reader how to label a boundary case. These are authored examples;
we are not claiming the rewrite improves Jev without running an evaluation.

| Vague | More testable | Edge case to include |
|---|---|---|
| Is this bad? | Does the reported bug prevent completion of the named task without a workaround? | Inconvenient but a workaround exists |
| Handle the refund | Does the message explicitly ask to return money already paid? | A price question is not a refund request |
| Choose the right team | Choose the primary queue using the stated precedence rules | A failed export of an invoice |
| Is the customer verified? | Does the supplied trusted account record have `verified=true`? | This is a code check, not a model question |

Write exclusions as well as inclusions. Keep each probability attached to its precise question;
changing the definition changes what the number means.

### 6. Demonstrate an order-sensitive pipeline
In a two-stage workflow, the first answer can choose which evidence to retrieve for the second.
Questions that depend on that retrieved evidence cannot be answered using the earlier, incomplete state.

In [5]:
documents = {
    'billing': 'Billing policy: duplicate charges can be investigated using a transaction ID.',
    'technical': 'Technical checklist: browser version, error text, and reproduction steps.',
}
first_stage_choice = 'technical'  # Hand-authored stand-in for a model result.
second_stage_state = {'message': 'CSV export crashes.', 'reference': documents[first_stage_choice]}
print(second_stage_state)
print('The second stage now has evidence that was absent from the first.')

{'message': 'CSV export crashes.', 'reference': 'Technical checklist: browser version, error text, and reproduction steps.'}
The second stage now has evidence that was absent from the first.


### 7. Design a fair comparison before collecting results

| Hold fixed | Record | Report |
|---|---|---|
| Task definitions and input cases | Model version, prompt version, dataset split | Accuracy and per-class errors |
| Allowed candidate set | Raw probabilities and answer validity | Calibration and log loss |
| Hardware/service conditions where possible | Input size, wall time, failures, retries | Median and p95 latency |
| Evaluation labels | Per-case usage and retry costs | Coverage, review rate, cost per resolved case |

Do not compare Jev's task probabilities with an LLM merely saying “I am 95% confident” as though
the two measurements have identical meaning. Benchmark the actual outputs and complete workflow.
Provider benchmarks are useful hypotheses to test; they do not replace measurements on your task.

**Predict first — exercise 4:** Your topic probability is 0.96, but required account permission is false.
Should the system act? Which part of the implementation should enforce the answer?

## Checks

In [6]:
assert [route(t) for t in tickets] == [
    'priority_technical_queue', 'billing_queue', 'human_review', 'collect_more_information']
assert np.isclose(lower, 0.5) and np.isclose(upper, 0.7)
assert total_cents == 3897
print('Routing examples, probability bounds, and exact arithmetic passed.')

Routing examples, probability bounds, and exact arithmetic passed.


## Next Steps
Add a “security” topic. Decide which deterministic rule should force review regardless of probability.
Then write down examples where two topics overlap, and decide how your criteria resolve them.

The final notebook translates these ideas into an [optional official API call](05_optional_real_jev_api.ipynb).

## Take three ideas with you

- Ask focused questions with clear criteria.
- Use code for arithmetic, permissions, and combining results.
- Dependent stages need the right evidence before they can run.

**You are ready to move on when:** You can decide which work belongs to the model and which rules belong in code.

## Check your understanding

Two questions run in parallel. Can I multiply their probabilities to get the probability that both are true?

Pause and explain your answer before opening the explanation.

<details>
<summary>Show the explanation</summary>
<p>Only if statistical independence is justified. Running calculations together does not make the events independent.</p>
</details>

For a short next step, try the [practice questions](../docs/PRACTICE.md).
For runnable exercises, use the [worked exercises](08_exercises_and_solutions.ipynb).

If you are unsure where to go next, use [the course outline](../docs/COURSE.md).

[Assignments](../assignments/README.md) · [Quick reference](../docs/QUICK_REFERENCE.md) · [Course outline](../docs/COURSE.md)